# Entrega 1: Proyecto Soluciones Intensivas en Datos (2026 -02)

# Notebook 3: Optiización del almacenamiento de los datos

## OceanWatch Analytics

### Presentado por: Grupo 2

#### Pablo Daniel Díaz Mera
#### Ana Catalina Gelvez
#### Christian Rene Mendez

## 1. Preparación para el almacenamiento

### Definición de espacio de trabajo

In [0]:
# Configuración del proyecto
CATALOGO = "workspace"
ESQUEMA = "oceanwatch"
VOLUMEN = "ais_data"

# Rutas de entrada
BASE = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"
RUTA_CSV = f"{BASE}/csv"

# Rutas para pruebas de almacenamiento
RUTA_ALMACENAMIENTO = f"{BASE}/almacenamiento"
RUTA_PARQUET = f"{RUTA_ALMACENAMIENTO}/parquet"
RUTA_DELTA = f"{RUTA_ALMACENAMIENTO}/delta"

### Verificación de archivos de entrada

In [0]:
archivos_csv = [
    archivo
    for archivo in dbutils.fs.ls(RUTA_CSV)
    if archivo.name.lower().endswith(".csv")
]

print(f"Archivos CSV encontrados: {len(archivos_csv)}")

for archivo in sorted(archivos_csv, key=lambda x: x.name):
    print(archivo.name)

Archivos CSV encontrados: 7
AIS_2023_06_01.csv
AIS_2023_06_02.csv
AIS_2023_06_03.csv
AIS_2023_06_04.csv
AIS_2023_06_05.csv
AIS_2023_06_06.csv
AIS_2023_06_07.csv


### Definición del esquema de la informacion de las AIS

In [0]:
from pyspark.sql import functions as f

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType
)

esquema_ais = StructType([
    StructField("MMSI", IntegerType(), True),
    StructField("BaseDateTime", TimestampType(), True),
    StructField("LAT", DoubleType(), True),
    StructField("LON", DoubleType(), True),
    StructField("SOG", DoubleType(), True),
    StructField("COG", DoubleType(), True),
    StructField("Heading", DoubleType(), True),
    StructField("VesselName", StringType(), True),
    StructField("IMO", StringType(), True),
    StructField("CallSign", StringType(), True),
    StructField("VesselType", IntegerType(), True),
    StructField("Status", IntegerType(), True),
    StructField("Length", DoubleType(), True),
    StructField("Width", DoubleType(), True),
    StructField("Draft", DoubleType(), True),
    StructField("Cargo", IntegerType(), True),
    StructField("TransceiverClass", StringType(), True)
])

### Validación del esquema definido

In [0]:
# Seleccionar únicamente el primer archivo CSV
ruta_prueba = f"{RUTA_CSV}/AIS_2023_06_01.csv"

# Leer utilizando el esquema definido
ais_prueba = (
    spark.read
    .option("header", "true")
    .schema(esquema_ais)
    .csv(ruta_prueba)
)

# Recuperar una muestra pequeña
muestra = ais_prueba.take(10)

# Visualizar los registros
display(spark.createDataFrame(muestra, schema=esquema_ais))

MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
367138690,2023-06-01T00:00:01.000Z,30.28335,-91.22283,0.0,273.0,511.0,ST FRANCISVILLE,null,WDD3968,60,0,43.0,18.0,null,60,A
368036960,2023-06-01T00:00:01.000Z,40.49818,-80.07596,3.3,118.6,131.0,GALE R RHODES,IMO8199788,WDK2590,0,12,20.0,7.0,2.6,0,A
368095340,2023-06-01T00:00:01.000Z,29.99678,-93.94959,0.0,309.8,511.0,GAMBLER,null,WDK8562,52,15,19.0,8.0,0.0,52,A
248301000,2023-06-01T00:00:03.000Z,26.13342,-80.11909,0.0,260.5,86.0,ATLAS,null,9HB5599,37,5,31.0,8.0,3.0,37,A
367186410,2023-06-01T00:00:03.000Z,30.00762,-93.95571,5.7,164.0,164.0,BRIAN O DANIELS,null,WDD7438,31,12,26.0,9.0,null,31,A
367108820,2023-06-01T00:00:05.000Z,37.82833,-76.27936,0.1,235.2,511.0,TIDELANDS,IMO7309443,WX5026,30,0,61.0,9.0,null,30,A
367320740,2023-06-01T00:00:05.000Z,38.73458,-82.87265,4.7,350.3,349.0,TENNESSEE,IMO8836584,WDE2124,31,0,42.0,12.0,null,52,A
368101220,2023-06-01T00:00:03.000Z,32.6922,-117.14808,0.0,360.0,511.0,WILLIAM F,null,WDK9183,33,15,40.0,18.0,0.0,33,A
367529860,2023-06-01T00:00:08.000Z,29.61437,-95.00393,0.2,197.6,285.0,C R MCCASKILL,null,WDG3808,90,3,67.0,null,null,33,A
367558580,2023-06-01T00:00:09.000Z,27.3326,-82.54605,0.0,233.2,221.0,DAYS LIKE THIS,null,WDG6586,37,0,40.0,7.0,null,37,A


### Visualización del esquema de AIS

In [0]:
ais = (
    spark.read
    .option("header", "true")
    .schema(esquema_ais)
    .csv(f"{RUTA_CSV}/*.csv")
)

ais.printSchema()

root
 |-- MMSI: integer (nullable = true)
 |-- BaseDateTime: timestamp (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- SOG: double (nullable = true)
 |-- COG: double (nullable = true)
 |-- Heading: double (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: integer (nullable = true)
 |-- Status: integer (nullable = true)
 |-- Length: double (nullable = true)
 |-- Width: double (nullable = true)
 |-- Draft: double (nullable = true)
 |-- Cargo: integer (nullable = true)
 |-- TransceiverClass: string (nullable = true)



## 2. Definición del propósito de consulta

- Se pretende optimizar la consulta de posiciones AIS por fecha y ubicación. De esta manera, se podrían identificar los registros de embarcaciones correspondientes a un tiempo específico y a un área delimitada por coordenadas

### Parámetros de la consulta

In [0]:
# Parámetros de la consulta objetivo
FECHA_CONSULTA = "2023-06-03"

# Zona geográfica de ejemplo
LAT_MIN = 25.0
LAT_MAX = 31.0
LON_MIN = -98.0
LON_MAX = -89.0

### Definición de la consulta de referencia

In [0]:
# Consulta de referencia sobre los CSV originales
consulta_csv = (
    ais
    .filter(
        (f.col("BaseDateTime") >= f.lit(FECHA_CONSULTA).cast("timestamp")) &
        (f.col("BaseDateTime") < f.date_add(
            f.lit(FECHA_CONSULTA).cast("date"), 1
        ).cast("timestamp"))
    )
    .filter(
        f.col("LAT").between(LAT_MIN, LAT_MAX) &
        f.col("LON").between(LON_MIN, LON_MAX)
    )
)

In [0]:
consulta_csv.explain(mode="formatted")

== Physical Plan ==
PhotonResultStage (5)
+- PhotonColumnarToRow (4)
   +- PhotonFilter (3)
      +- PhotonRowToColumnar (2)
         +- Scan csv  (1)


(1) Scan csv 
Output [17]: [MMSI#10246, BaseDateTime#10247, LAT#10248, LON#10249, SOG#10250, COG#10251, Heading#10252, VesselName#10253, IMO#10254, CallSign#10255, VesselType#10256, Status#10257, Length#10258, Width#10259, Draft#10260, Cargo#10261, TransceiverClass#10262]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/oceanwatch/ais_data/csv/AIS_2023_06_01.csv, ... 6 entries]
PushedFilters: [IsNotNull(BaseDateTime), IsNotNull(LAT), IsNotNull(LON), GreaterThanOrEqual(BaseDateTime,2023-06-03 00:00:00.0), LessThan(BaseDateTime,2023-06-04 00:00:00.0), GreaterThanOrEqual(LAT,25.0), LessThanOrEqual(LAT,31.0), GreaterThanOrEqual(LON,-98.0), LessThanOrEqual(LON,-89.0)]
ReadSchema: struct<MMSI:int,BaseDateTime:timestamp,LAT:double,LON:double,SOG:double,COG:double,Heading:double,VesselName:string,IMO:string,CallSign:string,V

- Se identifica que la consulta se puede realizar con estos parámetros.
- Se identifica que para hacer la identificación de los Vessels que cumplan los parámetros actualmente se haría la lectura de los 7 archivos .csv (Location: InMemoryFileIndex [dbfs:/Volumes/workspace/oceanwatch/ais_data/csv/AIS_2023_06_01.csv, ... 6 entries]
)

## 3. Almacenamiento en CSV

### Consulta de referencia sobre los archivos .csv

In [0]:
columnas_consulta = [
    "MMSI",
    "BaseDateTime",
    "LAT",
    "LON",
    "SOG"
]

consulta_csv_referencia = consulta_csv.select(*columnas_consulta)

In [0]:
import time

inicio = time.perf_counter()

total_posiciones = consulta_csv_referencia.count()

duracion = time.perf_counter() - inicio

print(f"Posiciones encontradas: {total_posiciones:,}")
print(f"Tiempo de ejecución: {duracion:.3f} segundos")

Posiciones encontradas: 2,127,150
Tiempo de ejecución: 22.891 segundos


- La consulta identificó aproximadamente 2,13 millones de posiciones AIS.
- El tiempo transcurrido fue de 19,075 segundos.
- La operación se ejecutó sobre la fuente CSV original, sin particionamiento Spark por fecha.
- Este resultado será nuestra línea base para comparar las alternativas de almacenamiento.

Se registra el resultado de la consulta de referencia ejecutada sobre los archivos CSV originales. Estos valores servirán como línea base para comparar las estrategias de almacenamiento Parquet y Delta Lake.

In [0]:
resultados_almacenamiento = [
    {
        "estrategia": "CSV original",
        "posiciones": total_posiciones,
        "tiempo_segundos": round(duracion, 3)
    }
]

display(resultados_almacenamiento)

estrategia,posiciones,tiempo_segundos
CSV original,2127150,22.891


## 4. Almacenamiento en parquet

Se almacena el conjunto completo de datos AIS en formato parquet, conservando el esquema y los registros originales. Esta primera versión no incorpora particionamiento por columnas, con el propósito de evaluar el efecto del formato columnar sobre el almacenamiento y las consultas. Los archivos CSV originales permanecen sin modificaciones.

In [0]:
import time

# Verificar si la ruta ya existe
try:
    dbutils.fs.ls(RUTA_PARQUET)
    ruta_existe = True
except Exception as e:
    if "FileNotFoundException" in str(e) or "PATH_NOT_FOUND" in str(e):
        ruta_existe = False
    else:
        raise

if ruta_existe:
    print("La ruta Parquet ya existe. No se realizará la escritura.")
else:
    inicio = time.perf_counter()

    (
        ais.write
        .mode("overwrite")
        .format("parquet")
        .save(RUTA_PARQUET)
    )

    tiempo_escritura_parquet = time.perf_counter() - inicio

    print(f"Ruta de almacenamiento: {RUTA_PARQUET}")
    print(f"Tiempo de escritura: {tiempo_escritura_parquet:.3f} segundos")

Ruta de almacenamiento: /Volumes/workspace/oceanwatch/ais_data/almacenamiento/parquet
Tiempo de escritura: 28.884 segundos


### Verificación del almacenamiento en parquet

In [0]:
# Identificar los archivos Parquet generados
archivos_parquet = [
    archivo
    for archivo in dbutils.fs.ls(RUTA_PARQUET)
    if archivo.name.endswith(".parquet")
]

# Calcular métricas de almacenamiento
cantidad_archivos = len(archivos_parquet)
tamano_total_bytes = sum(
    archivo.size for archivo in archivos_parquet
)

print("ALMACENAMIENTO PARQUET")
print("-" * 45)
print(f"Cantidad de archivos: {cantidad_archivos}")
print(f"Tamaño total: {tamano_total_bytes / (1024**3):.3f} GiB")
print(f"Tamaño promedio por archivo: {tamano_total_bytes / max(cantidad_archivos, 1) / (1024**2):.2f} MiB")

ALMACENAMIENTO PARQUET
---------------------------------------------
Cantidad de archivos: 49
Tamaño total: 1.741 GiB
Tamaño promedio por archivo: 36.38 MiB


La conversión de los datos AIS a formato Parquet se realizó conservando la estructura original de 17 columnas y generando 49 archivos distribuidos automáticamente por  Spark, con un tamaño total de 1,741 GiB y un tamaño promedio de 36,38 MiB por archivo. El proceso de escritura tomó 28,884 segundos.

### Lectura y verificación del esquema parquet

In [0]:
ais_parquet = spark.read.parquet(RUTA_PARQUET)

print(f"Columnas encontradas: {len(ais_parquet.columns)}")
ais_parquet.printSchema()

Columnas encontradas: 17
root
 |-- MMSI: integer (nullable = true)
 |-- BaseDateTime: timestamp (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- SOG: double (nullable = true)
 |-- COG: double (nullable = true)
 |-- Heading: double (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: integer (nullable = true)
 |-- Status: integer (nullable = true)
 |-- Length: double (nullable = true)
 |-- Width: double (nullable = true)
 |-- Draft: double (nullable = true)
 |-- Cargo: integer (nullable = true)
 |-- TransceiverClass: string (nullable = true)



la conversión a Parquet conservó correctamente la estructura del conjunto AIS. Sin embargo, esta verificación no confirma todavía que se hayan conservado todos los registros.

### Construcción de la consulta sobre parquet

In [0]:
consulta_parquet = (
    ais_parquet
    .filter(
        (f.col("BaseDateTime") >= "2023-06-03") &
        (f.col("BaseDateTime") < "2023-06-04") &
        (f.col("LAT").between(25.0, 31.0)) &
        (f.col("LON").between(-98.0, -89.0))
    )
    .select(
        "MMSI",
        "BaseDateTime",
        "LAT",
        "LON",
        "SOG"
    )
)

### Inspección del plan físico de la consulta sobre parquet

In [0]:
consulta_parquet.explain(mode="formatted")

== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [5]: [MMSI#10362, BaseDateTime#10363, LAT#10364, LON#10365, SOG#10366]
DictionaryFilters: [((BaseDateTime#10363 >= 2023-06-03 00:00:00) AND (BaseDateTime#10363 < 2023-06-04 00:00:00)), ((LON#10365 >= -98.0) AND (LON#10365 <= -89.0)), ((LAT#10364 >= 25.0) AND (LAT#10364 <= 31.0))]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/oceanwatch/ais_data/almacenamiento/parquet]
ReadSchema: struct<MMSI:int,BaseDateTime:timestamp,LAT:double,LON:double,SOG:double>
RequiredDataFilters: [isnotnull(BaseDateTime#10363), isnotnull(LAT#10364), isnotnull(LON#10365), (BaseDateTime#10363 >= 2023-06-03 00:00:00), (BaseDateTime#10363 < 2023-06-04 00:00:00), (LAT#10364 >= 25.0), (LAT#10364 <= 31.0), (LON#10365 >= -98.0), (LON#10365 <= -89.0)]

(2) PhotonColumnarToRow
Input [5]: [MMSI#10362, BaseDateTime#10363, LAT#10364, LON#10365, SOG#10366]

(3) PhotonResultStage
Input [5

Parquet presenta una ventaja técnica frente al CSV original: permite seleccionar únicamente las columnas requeridas y ofrece mecanismos adicionales para optimizar la lectura.

### Ejecución de la consulta sobre parquet

In [0]:
import time

inicio = time.perf_counter()

total_parquet = consulta_parquet.count()

duracion_parquet = time.perf_counter() - inicio

print("CONSULTA PARQUET")
print("-" * 45)
print(f"Posiciones encontradas: {total_parquet:,}")
print(f"Tiempo de ejecución: {duracion_parquet:.3f} segundos")
print(f"Coincide con CSV: {total_parquet == total_posiciones}")

CONSULTA PARQUET
---------------------------------------------
Posiciones encontradas: 2,127,150
Tiempo de ejecución: 1.828 segundos
Coincide con CSV: True


### Tabla comparativa de métricas sobre CSV y Parquet

In [0]:
resultados_almacenamiento.append({
    "estrategia": "Parquet sin particionamiento",
    "posiciones": total_parquet,
    "tiempo_segundos": round(duracion_parquet, 3)
})

display(spark.createDataFrame(resultados_almacenamiento))

estrategia,posiciones,tiempo_segundos
CSV original,2127150,22.891
Parquet sin particionamiento,2127150,1.828


El almacenamiento Parquet permitió ejecutar la consulta temporal y espacial en 1,828 segundos, frente a los 22,891 segundos registrados sobre CSV, manteniendo el mismo resultado de 2.127.150 posiciones. Esto representa una reducción aproximada del 92 % en el tiempo observado.

## 5. Almacenamiento en Delta

In [0]:
RUTA_DELTA = f"{RUTA_ALMACENAMIENTO}/delta"

print(f"Ruta de almacenamiento Delta: {RUTA_DELTA}")

Ruta de almacenamiento Delta: /Volumes/workspace/oceanwatch/ais_data/almacenamiento/delta


In [0]:
import time

# Verificar que la ruta Delta no exista previamente
try:
    dbutils.fs.ls(RUTA_DELTA)
    ruta_existe = True
except Exception as e:
    if "FileNotFoundException" in str(e) or "PATH_NOT_FOUND" in str(e):
        ruta_existe = False
    else:
        raise

if ruta_existe:
    print("La ruta Delta ya existe. No se realizará la escritura.")
else:
    inicio = time.perf_counter()

    (
        ais.write
        .format("delta")
        .mode("overwrite")
        .save(RUTA_DELTA)
    )

    tiempo_escritura_delta = time.perf_counter() - inicio

    print("ALMACENAMIENTO DELTA")
    print("-" * 45)
    print(f"Ruta: {RUTA_DELTA}")
    print(f"Tiempo de escritura: {tiempo_escritura_delta:.3f} segundos")

ALMACENAMIENTO DELTA
---------------------------------------------
Ruta: /Volumes/workspace/oceanwatch/ais_data/almacenamiento/delta
Tiempo de escritura: 29.987 segundos


Delta Lake presenta un tiempo de escritura similar al de Parquet en esta primera prueba. Todavía necesitamos evaluar su tamaño físico y rendimiento de consulta.

### Verificación del almacenamiento en Delta

In [0]:
# Inspeccionar el directorio Delta
archivos_delta = dbutils.fs.ls(RUTA_DELTA)

# Identificar archivos de datos Parquet
archivos_datos_delta = [
    archivo for archivo in archivos_delta
    if archivo.name.endswith(".parquet")
]

# Calcular métricas
cantidad_delta = len(archivos_datos_delta)
bytes_delta = sum(archivo.size for archivo in archivos_datos_delta)

print("ALMACENAMIENTO DELTA LAKE")
print("-" * 45)
print(f"Archivos de datos: {cantidad_delta}")
print(f"Tamaño de datos: {bytes_delta / (1024**3):.3f} GiB")
print(f"Tamaño promedio: {bytes_delta / max(cantidad_delta, 1) / (1024**2):.2f} MiB")
print(f"Directorio _delta_log presente: {any(a.name.rstrip('/') == '_delta_log' for a in archivos_delta)}")

ALMACENAMIENTO DELTA LAKE
---------------------------------------------
Archivos de datos: 52
Tamaño de datos: 1.741 GiB
Tamaño promedio: 34.28 MiB
Directorio _delta_log presente: True


### Verificación del esquema en Delta Lake

In [0]:
ais_delta = spark.read.format("delta").load(RUTA_DELTA)

print(f"Columnas encontradas: {len(ais_delta.columns)}")
ais_delta.printSchema()

Columnas encontradas: 17
root
 |-- MMSI: integer (nullable = true)
 |-- BaseDateTime: timestamp (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- SOG: double (nullable = true)
 |-- COG: double (nullable = true)
 |-- Heading: double (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: integer (nullable = true)
 |-- Status: integer (nullable = true)
 |-- Length: double (nullable = true)
 |-- Width: double (nullable = true)
 |-- Draft: double (nullable = true)
 |-- Cargo: integer (nullable = true)
 |-- TransceiverClass: string (nullable = true)



### Consulta de referencia sobre Delta Lake

In [0]:
# Consulta temporal y espacial sobre Delta Lake

consulta_delta = (
    ais_delta
    .filter(
        (f.col("BaseDateTime") >= "2023-06-03") &
        (f.col("BaseDateTime") < "2023-06-04") &
        f.col("LAT").between(25.0, 31.0) &
        f.col("LON").between(-98.0, -89.0)
    )
    .select("MMSI", "BaseDateTime", "LAT", "LON", "SOG")
)

# Examinar el plan físico de ejecución
consulta_delta.explain(mode="formatted")

== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [5]: [MMSI#10872, BaseDateTime#10873, LAT#10874, LON#10875, SOG#10876]
DictionaryFilters: [((LON#10875 >= -98.0) AND (LON#10875 <= -89.0)), ((LAT#10874 >= 25.0) AND (LAT#10874 <= 31.0)), ((BaseDateTime#10873 >= 2023-06-03 00:00:00) AND (BaseDateTime#10873 < 2023-06-04 00:00:00))]
Location: PreparedDeltaFileIndex [dbfs:/Volumes/workspace/oceanwatch/ais_data/almacenamiento/delta]
ReadSchema: struct<MMSI:int,BaseDateTime:timestamp,LAT:double,LON:double,SOG:double>
RequiredDataFilters: [isnotnull(BaseDateTime#10873), isnotnull(LAT#10874), isnotnull(LON#10875), (BaseDateTime#10873 >= 2023-06-03 00:00:00), (BaseDateTime#10873 < 2023-06-04 00:00:00), (LAT#10874 >= 25.0), (LAT#10874 <= 31.0), (LON#10875 >= -98.0), (LON#10875 <= -89.0)]

(2) PhotonColumnarToRow
Input [5]: [MMSI#10872, BaseDateTime#10873, LAT#10874, LON#10875, SOG#10876]

(3) PhotonResultStage
Input

### Medición del rendimiento en Delta lake

In [0]:
import time

# Medir tiempo de ejecución
inicio = time.perf_counter()

total_delta = consulta_delta.count()

duracion_delta = time.perf_counter() - inicio

# Presentar resultados
print("CONSULTA DELTA LAKE")
print("-" * 45)
print(f"Posiciones encontradas: {total_delta:,}")
print(f"Tiempo de ejecución: {duracion_delta:.3f} segundos")
print(f"Coincide con CSV: {total_delta == total_posiciones}")
print(f"Coincide con Parquet: {total_delta == total_parquet}")

CONSULTA DELTA LAKE
---------------------------------------------
Posiciones encontradas: 2,127,150
Tiempo de ejecución: 1.420 segundos
Coincide con CSV: True
Coincide con Parquet: True


## 6. Comparación consolidada de los tres formatos (CSV, Parquet y DeltaLake)

In [0]:
comparacion_final = [
    {
        "Formato": "CSV",
        "Archivos": len(archivos_csv),
        "Tamaño (GiB)": round(bytes_csv / (1024**3), 3),
        "Posiciones": total_posiciones,
        "Tiempo (s)": round(duracion, 3)
    },
    {
        "Formato": "Parquet",
        "Archivos": len(archivos_parquet),
        "Tamaño (GiB)": round(bytes_parquet / (1024**3), 3),
        "Posiciones": total_parquet,
        "Tiempo (s)": round(duracion_parquet, 3)
    },
    {
        "Formato": "Delta Lake",
        "Archivos": cantidad_delta,
        "Tamaño (GiB)": round(bytes_delta / (1024**3), 3),
        "Posiciones": total_delta,
        "Tiempo (s)": round(duracion_delta, 3)
    }
]

display(spark.createDataFrame(comparacion_final))

print("\nINDICADORES DE OPTIMIZACIÓN")
print("-" * 50)

for nombre, tamaño, tiempo in [
    ("Parquet", bytes_parquet, duracion_parquet),
    ("Delta Lake", bytes_delta, duracion_delta)
]:
    print(f"\n{nombre} frente a CSV:")
    print(f"Reducción de almacenamiento: {(1 - tamaño / bytes_csv) * 100:.2f}%")
    print(f"Reducción del tiempo: {(1 - tiempo / duracion) * 100:.2f}%")
    print(f"Factor de aceleración: {duracion / tiempo:.2f}x")

print("\nDelta Lake frente a Parquet:")
print(f"Variación del tiempo: {(1 - duracion_delta / duracion_parquet) * 100:.2f}% de reducción")

Archivos,Formato,Posiciones,Tamaño (GiB),Tiempo (s)
7,CSV,2127150,6.037,22.891
49,Parquet,2127150,1.741,1.828
52,Delta Lake,2127150,1.741,1.42



INDICADORES DE OPTIMIZACIÓN
--------------------------------------------------

Parquet frente a CSV:
Reducción de almacenamiento: 71.16%
Reducción del tiempo: 92.02%
Factor de aceleración: 12.53x

Delta Lake frente a CSV:
Reducción de almacenamiento: 71.16%
Reducción del tiempo: 93.80%
Factor de aceleración: 16.12x

Delta Lake frente a Parquet:
Variación del tiempo: 22.29% de reducción


- Tanto el almacenamiento en Parquet como en Delta Lake reducen aproximadamente un 71 % el espacio ocupado por los datos AIS. En Delta Lake todavía falta contabilizar el tamaño de _delta_log.
- Los tres formatos devolvieron exactamente 2.127.150 posiciones, lo que valida el resultado de la consulta de referencia.
- **Delta Lake es un candidato adecuado para el proyecto porque combina almacenamiento columnar con transacciones, versionado y operaciones de mantenimiento.**

## 7. Preparación del particionamiento

### Verificación de la distribución de los datos por día

In [0]:
distribucion_fechas = (
    ais_delta
    .withColumn("fecha", f.to_date("BaseDateTime"))
    .groupBy("fecha")
    .count()
    .orderBy("fecha")
)

display(distribucion_fechas)

fecha,count
2023-06-01,8808904
2023-06-02,9052241
2023-06-03,8036348
2023-06-04,8522645
2023-06-05,8613757
2023-06-06,8587938
2023-06-07,8911726


Esta distribución permite considerar el particionamiento por fecha como una estrategia candidata. Cada partición tendría millones de registros, evitando particiones excesivamente pequeñas.

### Definición del almacenamiento Delta Particionado

In [0]:
# Ruta para Delta Lake particionado por fecha
RUTA_DELTA_PARTICIONADO = (
    f"{RUTA_ALMACENAMIENTO}/delta_particionado_fecha"
)

print("CONFIGURACIÓN DEL PARTICIONAMIENTO")
print("-" * 50)
print(f"Ruta: {RUTA_DELTA_PARTICIONADO}")
print("Formato: Delta Lake")
print("Columna de particionamiento: fecha")
print("Granularidad: diaria")
print("Particiones esperadas: 7")

CONFIGURACIÓN DEL PARTICIONAMIENTO
--------------------------------------------------
Ruta: /Volumes/workspace/oceanwatch/ais_data/almacenamiento/delta_particionado_fecha
Formato: Delta Lake
Columna de particionamiento: fecha
Granularidad: diaria
Particiones esperadas: 7


### Escritura del DeltaLake particionado por fecha

In [0]:
import time

# Verificar que la ruta de salida no exista
try:
    dbutils.fs.ls(RUTA_DELTA_PARTICIONADO)
    ruta_existe = True
except Exception as e:
    if "FileNotFoundException" in str(e) or "PATH_NOT_FOUND" in str(e):
        ruta_existe = False
    else:
        raise

if ruta_existe:
    print("La ruta ya existe. No se realizará la escritura.")
else:
    # Crear la columna de particionamiento
    ais_delta_fecha = ais_delta.withColumn(
        "fecha",
        f.to_date("BaseDateTime")
    )

    inicio = time.perf_counter()

    (
        ais_delta_fecha.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("fecha")
        .save(RUTA_DELTA_PARTICIONADO)
    )

    tiempo_escritura_delta_particionado = (
        time.perf_counter() - inicio
    )

    print("ALMACENAMIENTO DELTA PARTICIONADO")
    print("-" * 50)
    print(f"Ruta: {RUTA_DELTA_PARTICIONADO}")
    print("Columna de particionamiento: fecha")
    print(
        f"Tiempo de escritura: "
        f"{tiempo_escritura_delta_particionado:.3f} segundos"
    )

ALMACENAMIENTO DELTA PARTICIONADO
--------------------------------------------------
Ruta: /Volumes/workspace/oceanwatch/ais_data/almacenamiento/delta_particionado_fecha
Columna de particionamiento: fecha
Tiempo de escritura: 15.815 segundos


La escritura particionada fue aproximadamente 47,26 % más rápida que la escritura Delta sin particionamiento.

### Verificación de las particiones

In [0]:
# ============================================================
# DISTRIBUCIÓN FÍSICA DE DELTA PARTICIONADO
# ============================================================

# Identificar directorios de partición
particiones = [
    archivo
    for archivo in dbutils.fs.ls(RUTA_DELTA_PARTICIONADO)
    if archivo.name.startswith("fecha=")
]

# Obtener métricas de cada partición
metricas_particiones = []

for particion in particiones:

    archivos = [
        archivo
        for archivo in dbutils.fs.ls(particion.path)
        if archivo.name.endswith(".parquet")
    ]

    bytes_particion = sum(archivo.size for archivo in archivos)

    metricas_particiones.append({
        "Fecha": particion.name.rstrip("/").replace("fecha=", ""),
        "Archivos": len(archivos),
        "Tamaño (GiB)": round(bytes_particion / (1024**3), 3),
        "Tamaño promedio (MiB)": round(
            bytes_particion / len(archivos) / (1024**2), 2
        ) if archivos else 0
    })

# Mostrar distribución por fecha
display(
    spark.createDataFrame(metricas_particiones)
    .orderBy("Fecha")
)

# Resumen general
total_archivos = sum(p["Archivos"] for p in metricas_particiones)
total_gib = sum(p["Tamaño (GiB)"] for p in metricas_particiones)

print("\nRESUMEN DEL ALMACENAMIENTO PARTICIONADO")
print("-" * 50)
print(f"Particiones encontradas: {len(particiones)}")
print(f"Total archivos Parquet: {total_archivos}")
print(f"Tamaño total aproximado: {total_gib:.3f} GiB")

Archivos,Fecha,Tamaño (GiB),Tamaño promedio (MiB)
1,2023-06-01,0.261,266.92
1,2023-06-02,0.269,275.4
1,2023-06-03,0.237,242.44
1,2023-06-04,0.25,256.32
1,2023-06-05,0.251,257.02
1,2023-06-06,0.25,256.43
1,2023-06-07,0.262,267.86



RESUMEN DEL ALMACENAMIENTO PARTICIONADO
--------------------------------------------------
Particiones encontradas: 7
Total archivos Parquet: 7
Tamaño total aproximado: 1.780 GiB


- Como era de esperarse, se encuentra una reducción del número de archivos ya que cuando no se definieron particiones, se generaron 52 archivos en Delta y ahora, definiendo particiones por fecha se generan sólo 7 (acordes a los días de los cuales se tiene información de AIS).
- El tamaño pasó de 1,741 GiB a aproximadamente 1,780 GiB, un incremento cercano al 2,24 %.
- Se puede tener un beneficio para las consultas debido a que como los archivos están separados por fecha, Spark podría leer únicamente la partición correspondiente al día solicitado.

### Consulta sobre Delta Lake particionado

In [0]:
# Leer Delta Lake particionado
ais_delta_particionado = (
    spark.read
    .format("delta")
    .load(RUTA_DELTA_PARTICIONADO)
)

# Consulta temporal y espacial
consulta_delta_particionado = (
    ais_delta_particionado
    .filter(
        (f.col("fecha") == "2023-06-03") &
        (f.col("BaseDateTime") >= "2023-06-03") &
        (f.col("BaseDateTime") < "2023-06-04") &
        f.col("LAT").between(25.0, 31.0) &
        f.col("LON").between(-98.0, -89.0)
    )
    .select("MMSI", "BaseDateTime", "LAT", "LON", "SOG")
)

# Examinar el plan físico
consulta_delta_particionado.explain(mode="formatted")

== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [6]: [MMSI#11591, BaseDateTime#11592, LAT#11593, LON#11594, SOG#11595, fecha#11608]
DictionaryFilters: [((BaseDateTime#11592 >= 2023-06-03 00:00:00) AND (BaseDateTime#11592 < 2023-06-04 00:00:00)), ((LAT#11593 >= 25.0) AND (LAT#11593 <= 31.0)), ((LON#11594 >= -98.0) AND (LON#11594 <= -89.0))]
Location: PreparedDeltaFileIndex [dbfs:/Volumes/workspace/oceanwatch/ais_data/almacenamiento/delta_particionado_fecha]
PartitionFilters: [isnotnull(fecha#11608), (fecha#11608 = 2023-06-03)]
ReadSchema: struct<MMSI:int,BaseDateTime:timestamp,LAT:double,LON:double,SOG:double>
RequiredDataFilters: [isnotnull(BaseDateTime#11592), isnotnull(LAT#11593), isnotnull(LON#11594), (BaseDateTime#11592 >= 2023-06-03 00:00:00), (BaseDateTime#11592 < 2023-06-04 00:00:00), (LAT#11593 >= 25.0), (LAT#11593 <= 31.0), (LON#11594 >= -98.0), (LON#11594 <= -89.0)]


- Se identifica que Databricks está aplicando partition pruning sobre la versión Delta Lake particionada por fecha. (PartitionFilters: [isnotnull(fecha#11608), (fecha#11608 = 2023-06-03)])

### Medición del rendimiento con particionamiento

In [0]:
import time

# Ejecutar consulta sobre Delta particionado
inicio = time.perf_counter()

total_delta_particionado = consulta_delta_particionado.count()

duracion_delta_particionado = time.perf_counter() - inicio

# Comparar con Delta sin particionar
print("CONSULTA DELTA PARTICIONADO")
print("-" * 50)

print(f"Posiciones encontradas: {total_delta_particionado:,}")
print(f"Tiempo de ejecución: {duracion_delta_particionado:.3f} segundos")

print(
    f"Coincide con Delta original: "
    f"{total_delta_particionado == total_delta}"
)

print("\nCOMPARACIÓN DE RENDIMIENTO")
print("-" * 50)

print(f"Delta sin particionar: {duracion_delta:.3f} segundos")
print(f"Delta particionado: {duracion_delta_particionado:.3f} segundos")

variacion = (
    (duracion_delta - duracion_delta_particionado)
    / duracion_delta
) * 100

print(f"Reducción del tiempo: {variacion:.2f}%")

CONSULTA DELTA PARTICIONADO
--------------------------------------------------
Posiciones encontradas: 2,127,150
Tiempo de ejecución: 1.300 segundos
Coincide con Delta original: True

COMPARACIÓN DE RENDIMIENTO
--------------------------------------------------
Delta sin particionar: 1.420 segundos
Delta particionado: 1.300 segundos
Reducción del tiempo: 8.47%


In [0]:
# Delta sin particionamiento
archivos_sin_particion = [
    archivo
    for archivo in dbutils.fs.ls(RUTA_DELTA)
    if archivo.name.endswith(".parquet")
]

bytes_sin_particion = sum(
    archivo.size for archivo in archivos_sin_particion
)

# Partición correspondiente a la fecha consultada
ruta_fecha = f"{RUTA_DELTA_PARTICIONADO}/fecha=2023-06-03/"

archivos_fecha = [
    archivo
    for archivo in dbutils.fs.ls(ruta_fecha)
    if archivo.name.endswith(".parquet")
]

bytes_fecha = sum(archivo.size for archivo in archivos_fecha)

# Tabla comparativa
comparacion_lectura = [
    {
        "Estrategia": "Delta sin particionar",
        "Archivos candidatos": len(archivos_sin_particion),
        "Tamaño candidato (MiB)": round(bytes_sin_particion / 1024**2, 2),
        "Tiempo (s)": round(duracion_delta, 3)
    },
    {
        "Estrategia": "Delta particionado",
        "Archivos candidatos": len(archivos_fecha),
        "Tamaño candidato (MiB)": round(bytes_fecha / 1024**2, 2),
        "Tiempo (s)": round(duracion_delta_particionado, 3)
    }
]

display(spark.createDataFrame(comparacion_lectura))

print("\nREDUCCIÓN DEL VOLUMEN FÍSICO CANDIDATO")
print("-" * 50)

reduccion_bytes = (
    1 - bytes_fecha / bytes_sin_particion
) * 100

print(f"Reducción: {reduccion_bytes:.2f}%")

Archivos candidatos,Estrategia,Tamaño candidato (MiB),Tiempo (s)
52,Delta sin particionar,1782.79,1.42
1,Delta particionado,242.44,1.3



REDUCCIÓN DEL VOLUMEN FÍSICO CANDIDATO
--------------------------------------------------
Reducción: 86.40%


A partir de las pruebas realizadas con los formatos CSV, Parquet y Delta Lake, se seleccionó Delta Lake con particionamiento diario como estrategia de almacenamiento para los datos AIS.

Esta decisión considera tanto la eficiencia del almacenamiento columnar como las funcionalidades transaccionales de Delta Lake, relevantes para la futura implementación de una arquitectura Medallion.

El particionamiento por fecha permitió organizar los 60.533.559 registros en siete particiones diarias. Para la consulta temporal y espacial evaluada, el volumen físico candidato se redujo un 86,40 % respecto al conjunto completo sin particionar, mientras que el tiempo de ejecución disminuyó un 8,47 %.

Aunque estas mediciones no permiten afirmar que el particionamiento será más rápido en todos los escenarios, los resultados respaldan su selección para las consultas diarias previstas en OceanWatch Analytics.

In [0]:
RUTA_DELTA_PARTICIONADO = "/Volumes/workspace/oceanwatch/ais_data/almacenamiento/delta_particionado_fecha"